In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-20b")

c:\ProgramData\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

In [4]:
model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")

response

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We need to call the get_weather function.', 'tool_calls': [{'id': 'fc_32bc769c-cf55-4b27-b696-5ddb1c3a5b79', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 127, 'total_tokens': 172, 'completion_time': 0.047128602, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.008609066, 'prompt_tokens_details': None, 'queue_time': 0.050043864, 'total_time': 0.055737668}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db988-4016-76d2-854b-d77e2ca8c9b1-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_32bc769c-cf55-4b27-b696-5ddb1c3a5b79', 'type': 'tool_call'}], 

In [6]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


In [7]:
messages = [{"role": "user", "content": "What's the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

In [8]:
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)

final_response.text

'It’s sunny in Boston.'

In [9]:
messages

[{'role': 'user', 'content': "What's the weather like in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. We must use the get_weather function.', 'tool_calls': [{'id': 'fc_a415eb96-00ac-4ce7-8f54-7779ec54b24e', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 127, 'total_tokens': 166, 'completion_time': 0.040250866, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.008526353, 'prompt_tokens_details': None, 'queue_time': 0.051223532, 'total_time': 0.048777219}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db988-40df-7550-9f61-1c36f117d319-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_a415eb96-00ac-4ce